# Group 11 — Milestone 2 / CPU-only benchmark (5 khoảng, một runtime)

Notebook này chạy benchmark M2 cho **cả 5 khoảng của nhóm, lần lượt trên cùng một runtime Colab CPU-only**, sau đó gộp bảng và tính số liệu cho cả nhóm. Muốn chạy một khoảng thì chỉ để lại một dòng trong `RANGES` (mục 4).

**Chạy hết từ trên xuống.** Thời gian ước tính:
- parse: khoảng 25 phút cho mỗi khoảng, tổng khoảng 2–2,5 giờ;
- cộng thêm giải nén, `fetch_bbl` và lưu lên Drive: khoảng 3–4 giờ.

Thời gian này vẫn dưới giới hạn 12 giờ của Colab.

**Cái gì được đo:** chỉ có `benchmark_m2.py` của từng khoảng (mục 8) và bước gộp `--export-only` (mục 9). Cài đặt, giải nén, `fetch_bbl`, test, validate, thống kê và lưu Drive đều **không** tính vào benchmark.

**Nếu Colab bị ngắt:** mỗi khoảng chạy xong được lưu lên Drive (`g11_m2_benchmark/<tên>_m2.tar.gz`, gồm `parsed/` và `logs/`). Mở runtime mới rồi chạy lại từ đầu: khoảng nào đã xong thì được khôi phục và bỏ qua, chỉ chạy tiếp các khoảng còn lại. Khoảng đang chạy dở lúc bị ngắt sẽ chạy lại từ đầu, không lấy số liệu dở dang. Lưu ý: các khoảng chạy ở runtime khác nhau có thể có CPU khác nhau. Mục 10 in CPU của từng khoảng để ghi vào book.

**Code lấy từ GitHub (`m2/main`).** Thay đổi nào chưa push thì Colab không thấy.

## 1. Xác minh môi trường CPU và thông tin phiên chạy

In [ ]:
import os, sys, re, shutil, platform, subprocess, json, csv, tarfile, datetime
from pathlib import Path
import psutil

CONTENT = Path("/content")

def cpu_model():
    try:
        for line in Path("/proc/cpuinfo").read_text().splitlines():
            if line.startswith("model name"):
                return line.split(":", 1)[1].strip()
    except OSError:
        pass
    return platform.processor() or "unknown"

def utc_now():
    return datetime.datetime.now(datetime.timezone.utc).isoformat()

print("UTC time:", utc_now())
print("Python:", sys.version)
print("CPU model:", cpu_model())
print("CPU logical:", psutil.cpu_count(logical=True))
print("Total RAM (GB):", round(psutil.virtual_memory().total / 1e9, 2))
print("Disk free (GB):", round(shutil.disk_usage(CONTENT).free / 1e9, 1))
print("Platform:", platform.platform())
print("COLAB_GPU env:", os.environ.get("COLAB_GPU", "not set"))
print("nvidia-smi path:", shutil.which("nvidia-smi"))
print("MANUAL CHECK REQUIRED: Runtime -> Change runtime type -> CPU")

## 2. Mount Google Drive – nơi chứa dữ liệu M1

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Clone đúng branch Milestone 2 đã push

In [ ]:
REPO_URL = "https://github.com/mituleefam/group11-arxiv-project.git"
BRANCH = "m2/main"  # branch chứa code đã được review
REPO_DIR = CONTENT / "group11-arxiv-project"
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)], check=True)
actual_branch = subprocess.check_output(["git", "-C", str(REPO_DIR), "branch", "--show-current"], text=True).strip()
assert actual_branch == BRANCH, f"Expected {BRANCH}, found {actual_branch}; reopen with a clean runtime"
COMMIT = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
print("Branch:", actual_branch)
print("Commit:", COMMIT)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO_DIR / "src" / "requirements.txt")], check=True)
import pyarrow
print("pyarrow:", pyarrow.__version__)

## 4. Cấu hình: các khoảng và nơi lưu kết quả

Mỗi khoảng trỏ tới dữ liệu M1 của khoảng đó trên Drive, ở một trong 3 dạng:
1. thư mục có `corpus/`, và nếu có thì thêm `work/paper_manifest.jsonl`;
2. một file `.tar.gz`, ví dụ `Tram_MS2.tar.gz`;
3. một list nhiều `.tar.gz`, được giải nén theo đúng thứ tự trong list (file sau ghi đè file trước).

Dữ liệu gốc trên Drive chỉ được đọc, không bị sửa. Notebook chỉ xử lý bản copy trong `/content`.

Nếu người chạy chưa có dữ liệu của các thành viên khác trên Drive của mình, mỗi thành viên chia sẻ file của mình, người chạy bấm *Add shortcut to Drive*, rồi trỏ tới shortcut đó.

In [ ]:
DRIVE_ROOT = CONTENT / "drive" / "MyDrive"

# SỬA: tên khoảng -> dữ liệu M1 của khoảng đó. Chỉ chạy một khoảng: để lại đúng một dòng.
RANGES = {
    "Tram": DRIVE_ROOT / "EDIT_ME" / "Tram_MS2.tar.gz",   # 2511.21709 – 2512.03230
    "Thu":  DRIVE_ROOT / "EDIT_ME" / "Thu_MS2.tar.gz",    # 2512.13231 – 2512.18230
    "Phat": DRIVE_ROOT / "EDIT_ME" / "Phat_MS2.tar.gz",   # 2601.13156 – 2601.18155
    "Bao":  DRIVE_ROOT / "EDIT_ME" / "GBao_MS2.tar.gz",   # 2604.05535 – 2604.10534
    "Quan": DRIVE_ROOT / "EDIT_ME" / "Quan_MS2.tar.gz",   # 2605.12338 – 2605.17337
}
EXPECTED_PER_RANGE = 5000   # số ID được giao cho mỗi khoảng
WORKERS = 2                 # giữ nguyên cho mọi khoảng để số liệu so sánh được
RUN_FETCH_BBL = True        # tải bù .bbl (cần mạng, không tính vào benchmark)
SAVE_MERGED_TABLES = True   # lưu 3 bảng Parquet đã gộp lên Drive (dùng cho M3 và Hugging Face)

WORK_ROOT = CONTENT / "m2_work"                 # mỗi khoảng một thư mục: m2_work/<tên>
RESULT_DRIVE = DRIVE_ROOT / "g11_m2_benchmark"  # kết quả từng khoảng + kết quả gộp
BBL_DRIVE_DIR = DRIVE_ROOT / "m2_bbl"           # .bbl đã tải, cùng chỗ với notebook cũ
MERGED = WORK_ROOT / "_merged"
MERGED_TABLES = MERGED / "tables"
EXPORT_LOGS = MERGED / "logs"
GROUP_OUT = MERGED / "group_report"

def sources(name):
    value = RANGES[name]
    return [Path(v) for v in value] if isinstance(value, (list, tuple)) else [Path(value)]

def work_dir(name):
    return WORK_ROOT / name

def is_done(name):
    return (work_dir(name) / "m2_done.json").exists()

def sh(cmd, check=True):
    cmd = [str(c) for c in cmd]
    print("$", " ".join(cmd), flush=True)
    return subprocess.run(cmd, cwd=REPO_DIR, check=check)

assert RANGES, "RANGES is empty"
for name in RANGES:
    assert re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]*", name), f"Bad range name: {name}"
    for source in sources(name):
        assert source.exists(), f"{name}: missing M1 data source {source}"
        assert not WORK_ROOT.resolve().is_relative_to(source.resolve()), "Never work inside the Drive data"
print("Ranges:", ", ".join(RANGES))
print("Results on Drive:", RESULT_DRIVE)

## 5. Nạp corpus M1 của từng khoảng vào `/content`

Với mỗi khoảng: giải nén vào `m2_work/<tên>/`, bỏ các `parsed/` còn sót, chép `work/paper_manifest.jsonl`. Nếu khoảng đó đã chạy xong ở một runtime trước, kết quả (`parsed/`, `logs/`) được khôi phục từ Drive và khoảng đó sẽ được bỏ qua ở mục 8.

In [ ]:
def load_range(name):
    d = work_dir(name)
    if (d / ".loaded").exists():
        print(f"{name}: already in {d}")
        return
    assert not d.exists(), f"{d} is left over from an interrupted load: delete this folder and rerun the cell"
    srcs = sources(name)
    extract = WORK_ROOT / "_extract" / name
    shutil.rmtree(extract, ignore_errors=True)
    extract.mkdir(parents=True)
    if srcs[0].is_dir():
        assert len(srcs) == 1, f"{name}: a folder source must be alone"
        root = srcs[0]
    else:
        for archive in srcs:
            assert archive.name.endswith((".tar.gz", ".tgz")), f"Expected a .tar.gz archive: {archive}"
            print(f"{name}: extracting {archive.name}", flush=True)
            with tarfile.open(archive, mode="r:gz") as tar:
                tar.extractall(extract, filter="data")  # ignore symlinks and unsafe paths
        found = [folder.parent for folder in extract.rglob("corpus") if folder.is_dir()]
        assert len(found) == 1, f"{name}: expected exactly one corpus folder; found {found}"
        root = found[0]
    assert (root / "corpus").is_dir(), f"Input folder missing: {root / 'corpus'}"
    d.mkdir(parents=True)
    (d / "work").mkdir()
    if root == srcs[0]:  # a folder on Drive: copy it, never move
        shutil.copytree(root / "corpus", d / "corpus", ignore=shutil.ignore_patterns("parsed"))
    else:                # our own extracted copy: moving it saves disk and time
        shutil.move(str(root / "corpus"), str(d / "corpus"))
        for parsed in (d / "corpus").glob("*/parsed"):
            shutil.rmtree(parsed)
    manifest = root / "work" / "paper_manifest.jsonl"
    if manifest.exists():
        shutil.copy2(manifest, d / "work" / "paper_manifest.jsonl")
    else:
        print(f"WARNING {name}: no work/paper_manifest.jsonl, every folder of corpus/ is parsed")
    shutil.rmtree(extract, ignore_errors=True)
    count = sum(p.is_dir() for p in (d / "corpus").iterdir())
    assert count > 0, f"{name}: no paper folders found"
    (d / ".loaded").write_text(json.dumps({"sources": [str(s) for s in srcs], "loaded_utc": utc_now(),
                                           "paper_folders": count}), encoding="utf-8")
    print(f"{name}: {count} paper folders in {d}")

def restore_range(name):
    archive = RESULT_DRIVE / f"{name}_m2.tar.gz"
    if is_done(name) or not archive.exists():
        return
    with tarfile.open(archive, mode="r:gz") as tar:
        tar.extractall(work_dir(name), filter="data")
    print(f"{name}: restored the finished run from {archive}; mục 8 sẽ bỏ qua khoảng này")

for name in RANGES:
    load_range(name)
    restore_range(name)
print("Done:", [n for n in RANGES if is_done(n)], "| To run:", [n for n in RANGES if not is_done(n)])
print("Disk free (GB):", round(shutil.disk_usage(CONTENT).free / 1e9, 1))

## 6. Tải bù file `.bbl` còn thiếu

Bài chỉ có `.bbl` mà không có `.bib` sẽ mất tham khảo nếu thiếu file này, và lần crawl đầu chưa giữ `.bbl`. Cell này chỉ tải những version có trích dẫn mà không có nguồn tham khảo nào, và chỉ ghi vào bản copy trong `/content`.

Sau mỗi đợt, các file `.bbl` của khoảng được lưu thành một file nén `m2_bbl/<tên file dữ liệu>.tar.gz` trên Drive. Runtime mới sẽ khôi phục từ file này và tải tiếp, không tải lại từ đầu. Nếu một thành viên đã tải `.bbl` bằng notebook cũ trên Drive của họ, chép file đó vào `m2_bbl/` của người chạy để khỏi tải lại.

Bước này cần mạng và **không** tính vào benchmark. Dữ liệu đã có đủ `.bbl` thì cell báo 0 version và kết thúc ngay.

In [ ]:
def bbl_archive(name):
    srcs = sources(name)
    stem = srcs[0].name.split(".")[0] if len(srcs) == 1 else name
    return BBL_DRIVE_DIR / f"{stem}.tar.gz"

def save_bbl(name):
    """Các file .bbl, versions.json của bài có .bbl và log của một khoảng, gom thành một file nén trên Drive."""
    d = work_dir(name)
    temporary = CONTENT / "m2_bbl.tar.gz"
    with tarfile.open(temporary, "w:gz") as archive:
        papers = set()
        for bbl in (d / "corpus").rglob("*.bbl"):
            archive.add(bbl, arcname=str(bbl.relative_to(d)))
            papers.add(bbl.relative_to(d / "corpus").parts[0])
        for paper in sorted(papers):
            archive.add(d / "corpus" / paper / "versions.json", arcname=f"corpus/{paper}/versions.json")
        log = d / "logs" / "bbl_log.csv"
        if log.exists():
            archive.add(log, arcname="logs/bbl_log.csv")
    BBL_DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    shutil.copy2(temporary, bbl_archive(name))
    temporary.unlink()

def fetch_bbl(name):
    d, saved = work_dir(name), bbl_archive(name)
    if saved.exists():
        with tarfile.open(saved) as archive:
            archive.extractall(d, filter="data")
        print(f"{name}: restored {saved}")
    fetch = [sys.executable, "src/fetch_bbl.py", "--data-dir", d]
    sh(fetch + ["--dry-run"])
    while True:  # từng đợt 300 bài, lưu lên Drive sau mỗi đợt để Colab có ngắt cũng không mất
        cmd = [str(c) for c in fetch + ["--limit", "300"]]
        run = subprocess.run(cmd, cwd=REPO_DIR, capture_output=True, text=True)
        print("\n".join(run.stdout.splitlines()[-2:]), run.stderr[-500:])
        assert run.returncode in (0, 1), (f"{name}: fetch_bbl.py dừng với mã {run.returncode}, chạy lại cell này "
                                          "(mã 2: đợi khóa crawler.lock hết hạn, tối đa 3 phút)")
        save_bbl(name)
        if not re.search(r"(?:added|no_bbl|not_found)=", run.stdout):
            break  # không còn gì để tải, hoặc chỉ còn version lỗi: chạy lại cell này để thử lại chúng

if RUN_FETCH_BBL:
    for name in RANGES:
        if is_done(name):
            print(f"{name}: finished earlier, skip")
            continue
        print(f"===== {name}")
        fetch_bbl(name)

## 7. Chạy unit test, rồi chạy thử 5 bài để kiểm tra contract

In [ ]:
sh([sys.executable, "src/test_m2_bbl.py"])
sh([sys.executable, "src/test_m2_member2.py"])
sh([sys.executable, "-m", "unittest", "discover", "-s", "src", "-p", "test_m2_member4.py", "-v"])
sh([sys.executable, "src/test_m2_member1.py"])
sh([sys.executable, "src/reference_parser/test_m2_member3.py"])
sh([sys.executable, "src/test_m2_full_fixes.py"])
sh([sys.executable, "src/test_m2_case_diagnostics.py"])
sh([sys.executable, "src/test_m2_warning_audit.py"])

todo = [n for n in RANGES if not is_done(n)]
if todo:
    trial = work_dir(todo[0])
    sh([sys.executable, "src/pipeline_m2.py", "--data-dir", trial, "--limit", "5", "--workers", WORKERS, "--force"])
    sh([sys.executable, "src/validate_m2.py", "--data-dir", trial, "--limit", "5", "--require-parquet"])
    print("Check the output: do NOT accept a full-group benchmark if Member 2/3 stand-ins remain.")

## 8. Benchmark chính thức – parse full từng khoảng, lần lượt

Với mỗi khoảng chưa xong, cell này làm các bước sau:
1. Xóa `parsed/`, `tables/` và `logs/` **trong bản copy `m2_work/<tên>`**. Không đụng vào `.tex`, `.bib`, `.bbl`, cũng không đụng vào Drive.
2. Chạy `benchmark_m2.py --force`. **Chỉ bước này được đo.**
3. Chạy validate, `reference_stats` và `statistics_m2`.
4. Ghi `m2_done.json` (commit, CPU, wall time, RAM) rồi lưu `parsed/` và `logs/` của khoảng lên Drive.

Nếu một bước báo lỗi, ví dụ validate có FAIL, cell dừng ở khoảng đó và khoảng đó không được đánh dấu là xong. Sửa lỗi rồi chạy lại cell: các khoảng đã xong được giữ nguyên.

In [ ]:
def save_range(name):
    d = work_dir(name)
    temporary = CONTENT / f"{name}_m2.tar.gz"
    with tarfile.open(temporary, "w:gz") as archive:
        for parsed in sorted((d / "corpus").glob("*/parsed")):
            archive.add(parsed, arcname=str(parsed.relative_to(d)))
        archive.add(d / "logs", arcname="logs")
        if (d / "tables" / "tables.json").exists():
            archive.add(d / "tables" / "tables.json", arcname="tables/tables.json")
        archive.add(d / "m2_done.json", arcname="m2_done.json")  # last: its presence means complete
    RESULT_DRIVE.mkdir(parents=True, exist_ok=True)
    shutil.copy2(temporary, RESULT_DRIVE / temporary.name)
    temporary.unlink()
    print(f"{name}: saved to {RESULT_DRIVE / temporary.name}")

for name in RANGES:
    d = work_dir(name)
    if is_done(name):
        print(f"{name}: finished earlier, skip")
        continue
    print(f"===== {name}", flush=True)
    assert d.parent == WORK_ROOT and (d / "corpus").is_dir()
    for folder in (d / "corpus").iterdir():
        if folder.is_dir():
            shutil.rmtree(folder / "parsed", ignore_errors=True)
    shutil.rmtree(d / "tables", ignore_errors=True)
    shutil.rmtree(d / "logs", ignore_errors=True)
    sh([sys.executable, "src/benchmark_m2.py", "--data-dir", d, "--workers", WORKERS, "--force",
        "--interval", "0.5", "--disk-interval", "30",
        "--environment", "google_colab", "--cpu-only-declared"])
    sh([sys.executable, "src/validate_m2.py", "--data-dir", d, "--require-parquet"])
    sh([sys.executable, "src/reference_parser/reference_stats.py", "--data-dir", d])
    sh([sys.executable, "src/statistics_m2.py", "--data-dir", d])
    with (d / "logs" / "performance_log_m2.csv").open(encoding="utf-8-sig", newline="") as f:
        run = list(csv.DictReader(f))[-1]
    assert run["status"] == "finished" and run["exit_code"] == "0", run
    done = {"range": name, "finished_utc": utc_now(), "commit": COMMIT, "cpu_model": cpu_model(),
            "cpu_logical": psutil.cpu_count(logical=True), "workers": WORKERS, "run_id": run["run_id"],
            "wall_seconds": float(run["wall_seconds"]),
            "ram_sampled_peak_bytes": float(run["ram_sampled_peak_bytes"]),
            "parquet_final_bytes": float(run["parquet_final_bytes"])}
    (d / "m2_done.json").write_text(json.dumps(done, indent=2), encoding="utf-8")
    save_range(name)
    print(f"{name}: wall {done['wall_seconds'] / 60:.1f} min, peak RAM {done['ram_sampled_peak_bytes'] / 1e9:.2f} GB")

## 9. Gộp bảng của cả nhóm (`--export-only`, có đo)

`pipeline_m2.py --export-only` đọc `parsed/` của tất cả các khoảng và ghi ra 3 bảng Parquet chung. Bước này chạy dưới `benchmark.py`. File `performance_summary_*.json` mà nó tạo ra chính là `--export-summary` mà mục 10 cần. Sau khi gộp, cell kiểm tra số dòng của bảng gộp có đúng bằng tổng số dòng của các khoảng hay không.

In [ ]:
pending = [n for n in RANGES if not is_done(n)]
assert not pending, f"Run mục 8 first; not finished: {pending}"
shutil.rmtree(MERGED_TABLES, ignore_errors=True)
shutil.rmtree(EXPORT_LOGS, ignore_errors=True)
EXPORT_LOGS.mkdir(parents=True)
sh([sys.executable, "src/benchmark.py", "--data-dir", MERGED_TABLES, "--output-dir", EXPORT_LOGS,
    "--interval", "0.5", "--disk-interval", "30", "--",
    sys.executable, "src/pipeline_m2.py", "--data-dir", *[work_dir(n) for n in RANGES],
    "--export-only", "--out", MERGED_TABLES])
summaries = sorted(EXPORT_LOGS.glob("performance_summary_*.json"))
assert len(summaries) == 1, summaries
EXPORT_SUMMARY = summaries[0]

merged = json.loads((MERGED_TABLES / "tables.json").read_text(encoding="utf-8"))
parts = [json.loads((work_dir(n) / "tables" / "tables.json").read_text(encoding="utf-8")) for n in RANGES]
for key in ("papers", "elements_rows", "hierarchy_rows", "bibtex_rows"):
    total = sum(p.get(key, 0) for p in parts)
    assert merged.get(key) == total, f"{key}: merged={merged.get(key)}, sum of ranges={total}"
    print(f"{key}: {merged[key]:,}")
print("Export summary:", EXPORT_SUMMARY)

## 10. Số liệu benchmark của cả nhóm

Cell này chạy `group_benchmark.py --m2` với các quy tắc sau:
- **wall time** = tổng wall time của các khoảng + thời gian bước gộp;
- **RAM đỉnh** = giá trị lớn nhất, có tính cả bước gộp;
- **dung lượng** = kích thước các bảng đã gộp;
- **tỉ lệ dedup** tính lại từ tổng số trước và sau dedup.

Mỗi lần chạy của một khoảng đã tự export bảng riêng của khoảng đó, nên tổng wall time hơi cao hơn một lần chạy liền. Nếu có `WARNING`, đọc kỹ trước khi đưa số liệu vào book. Một khoảng có ít bài hơn `EXPECTED_PER_RANGE`, vì M1 không tìm thấy một số ID, cũng sinh cảnh báo.

In [ ]:
result = sh([sys.executable, "src/group_benchmark.py", "--m2", *[f"{n}={work_dir(n)}" for n in RANGES],
             "--expected-per-member", EXPECTED_PER_RANGE, "--tables-dir", MERGED_TABLES,
             "--export-summary", EXPORT_SUMMARY, "--out", GROUP_OUT], check=False)
assert result.returncode in (0, 2), f"group_benchmark.py failed with code {result.returncode}"
report = json.loads((GROUP_OUT / "group_benchmark_m2.json").read_text(encoding="utf-8"))

def fmt(value, scale, digits):
    return "-" if value is None else f"{value / scale:.{digits}f}"

print(f"{'range':<8}{'papers':>8}{'wall h':>9}{'RAM GB':>9}{'Parquet GB':>12}{'elem dedup %':>14}{'bib dedup %':>13}  CPU")
for row in report["members"] + [report["group"]]:
    name = row["member"]
    cpu = (json.loads((work_dir(name) / "m2_done.json").read_text())["cpu_model"]
           if name in RANGES else "")
    print(f"{name:<8}{fmt(row['papers'], 1, 0):>8}{fmt(row['wall_seconds'], 3600, 2):>9}"
          f"{fmt(row['ram_peak_bytes'], 1e9, 2):>9}{fmt(row['parquet_final_bytes'], 1e9, 3):>12}"
          f"{fmt(row['element_reduction_percent'], 1, 2):>14}{fmt(row['bibtex_reduction_percent'], 1, 2):>13}  {cpu}")
group = report["group"]
if group["wall_seconds"] and group["papers"]:
    print(f"\nAverage per paper: {group['wall_seconds'] / group['papers']:.3f} s")
for warning in report["warnings"]:
    print("WARNING:", warning)
print("\nGROUP:", group["completeness"])

## 11. Lưu kết quả gộp lên Drive (không commit corpus hay bảng Parquet lớn lên GitHub)

In [ ]:
dest = RESULT_DRIVE / "_merged"
dest.mkdir(parents=True, exist_ok=True)
shutil.copytree(GROUP_OUT, dest / "group_report", dirs_exist_ok=True)
shutil.copytree(EXPORT_LOGS, dest / "export_logs", dirs_exist_ok=True)
for name in RANGES:
    shutil.copy2(work_dir(name) / "m2_done.json", dest / f"{name}_m2_done.json")
if SAVE_MERGED_TABLES:
    shutil.copytree(MERGED_TABLES, dest / "tables", dirs_exist_ok=True)
else:
    shutil.copy2(MERGED_TABLES / "tables.json", dest / "tables.json")
print("Saved:", dest)
print("Do not push these files or the raw corpus to the GitHub source repository.")